In [1]:
import os
import shutil
import pandas as pd
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset, random_split
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(224, padding=16),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


In [3]:
# ImageNet normalization

train_transform_tl = transforms.Compose([
    transforms.RandomCrop(224, padding=16),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])


In [4]:
# ImageNet Normalization

eval_transform_tl = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [5]:
# Load dataset WITHOUT transforms (for indexing only)
data_dir = r"Harvest_Data/FRUIT_BINARY"

base_dataset = datasets.ImageFolder(
    root=data_dir,
    transform=None
)

samples = base_dataset.samples  # [(path, label), ...]


In [6]:
from collections import defaultdict
import os

fruit_to_indices = defaultdict(list)

for idx, (path, label) in enumerate(samples):
    filename = os.path.basename(path)
    
    # Example: F_Mango_1.jpg → Mango
    fruit_name = filename.split("_")[1]
    
    fruit_to_indices[(fruit_name, label)].append(idx)


In [7]:
fruit_to_indices.keys()

dict_keys([('Banana', 0), ('Lemon', 0), ('Lulo', 0), ('Mango', 0), ('Orange', 0), ('Strawberry', 0), ('Tamarillo', 0), ('Tomato', 0), ('Banana', 1), ('Lemon', 1), ('Lulo', 1), ('Mango', 1), ('Orange', 1), ('Strawberry', 1), ('Tamarillo', 1), ('Tomato', 1)])

In [8]:
from sklearn.model_selection import train_test_split

train_idx, val_idx, test_idx = [], [], []

for (fruit, label), indices in fruit_to_indices.items():
    
    # 70% train, 15% val, 15% test
    tr, temp = train_test_split(
        indices,
        test_size=0.30,
        random_state=42
    )
    
    val, te = train_test_split(
        temp,
        test_size=0.50,
        random_state=42
    )
    
    train_idx.extend(tr)
    val_idx.extend(val)
    test_idx.extend(te)


In [9]:
train_dataset = datasets.ImageFolder(
    root=data_dir,
    transform=train_transform_tl
)

val_dataset = datasets.ImageFolder(
    root=data_dir,
    transform=eval_transform_tl
)

test_dataset = datasets.ImageFolder(
    root=data_dir,
    transform=eval_transform_tl
)


In [11]:
from torch.utils.data import Subset

train_subset = Subset(train_dataset, train_idx)
val_subset   = Subset(val_dataset, val_idx)
test_subset  = Subset(test_dataset, test_idx)

In [12]:
import os
from torch.utils.data import DataLoader

NUM_WORKERS = min(4, os.cpu_count())

train_loader = DataLoader(
    train_subset,
    batch_size=32,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_subset,
    batch_size=32,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


In [13]:
print("Train:", len(train_subset))
print("Val:", len(val_subset))
print("Test:", len(test_subset))

Train: 11200
Val: 2400
Test: 2400


In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
class BaseCNN(nn.Module):
    def __init__(self, num_classes):
        super(BaseCNN, self).__init__()

        self.network = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3,3), padding='same'), # (16, 224, 224)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)), # (16, 112, 112)

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3,3), padding='same'), # (32, 112, 112)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)), # (32, 56, 56)

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3,3), padding='same'), # (64, 56, 56)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)), # (64, 28, 28)

            # nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3,3), padding='same'), # (128, 28, 28)
            # nn.ReLU(),
            # nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)) # (128, 14, 14)
        
        )

        self.flatten = nn.Flatten()

        self.fc_layer = nn.Sequential(
            nn.Linear(64*28*28, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )


    def forward(self, x):
        x = self.network(x)
        x = self.flatten(x)
        x = self.fc_layer(x)

        return x            

In [ ]:
model = BaseCNN(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) # Common defaults: betas=(0.9, 0.999), eps=1e-8
print(model)

#### Training & Validation function

In [15]:
def train_and_validate(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs
):
    """
    Trains and validates the model.
    Prints train loss and validation loss every epoch.
    """

    assert epochs % 1 == 0, "Epochs must be in steps of 1"

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # --------------------
        # Training phase
        # --------------------
        model.train()
        running_train_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --------------------
        # Validation phase (every epoch)
        # --------------------
        model.eval()
        running_val_loss = 0.0
        total = 0
        correct = 0

        true_labels = []
        pred_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item()

                _, predicted = torch.max(outputs, dim=1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                true_labels.extend(labels.cpu().numpy())
                pred_labels.extend(predicted.cpu().numpy())

        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        accuracy = 100.0 * correct / total

        # --------------------
        # Logging every epoch
        # --------------------
        if (epoch + 1) % 1 == 0:
            print(
                f"Epoch [{epoch+1}/{epochs}] | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val Acc: {accuracy:.2f}%"
            )

    return train_losses, val_losses, true_labels, pred_labels


In [ ]:
train_losses, val_losses, true_labels, pred_labels = train_and_validate(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=15
)


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, pred_labels)

# class_names = full_dataset.classe  # class names
class_names = ["fresh", "spoiled"]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,  # we want numbers in cells
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
model.eval()

test_loss = 0.0
correct = 0
total = 0

true_labels = []
pred_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # store for confusion matrix / analysis
        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)
test_accuracy = 100.0 * correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")


In [ ]:
import torch

MODEL_PATH = "fruits_freshness_simple_classifier.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "class_to_idx": train_dataset.class_to_idx
}, MODEL_PATH)

print("Model saved successfully!")


#### Plot train & validation losses

In [ ]:
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 6))

plt.plot(epochs, train_losses, label="Training Loss", marker="o")
plt.plot(epochs, val_losses, label="Validation Loss", marker="o")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

#### Model 2 - Using global average pooling

In [ ]:
class NextCNN(nn.Module):
    def __init__(self, num_classes):
        super(NextCNN, self).__init__()

        self.network = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3,3), padding='same'), # (16, 224, 224)
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)), # (16, 112, 112)

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3,3), padding='same'), # (32, 112, 112)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)), # (32, 56, 56)

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3,3), padding='same'), # (64, 56, 56)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)), # (64, 28, 28)

            )

        #self.flatten = nn.Flatten()

        # Global average pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.fc_layer = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, num_classes),          

        )


    def forward(self, x):
        x = self.network(x)
        x = self.gap(x)
        x = torch.flatten(x,1)
        x = self.fc_layer(x)

        return x            

In [ ]:
model2 = NextCNN(num_classes=2).to(device)
criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(model2.parameters(), lr=0.001, weight_decay=1e-4) # Common defaults: betas=(0.9, 0.999), eps=1e-8
print(model2)

In [ ]:
train_losses2, val_losses2, true_labels2, pred_labels2 = train_and_validate(
    model=model2,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion2,
    optimizer=optimizer2,
    device=device,
    epochs=15
)


In [ ]:
model2.eval()

test_loss2 = 0.0
correct2 = 0
total2 = 0

true_labels2 = []
pred_labels2 = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model2(images)
        loss = criterion2(outputs, labels)

        test_loss2 += loss.item()

        _, predicted = torch.max(outputs, dim=1)

        total2 += labels.size(0)
        correct2 += (predicted == labels).sum().item()

        # store for confusion matrix
        true_labels2.extend(labels.cpu().numpy())
        pred_labels2.extend(predicted.cpu().numpy())

avg_test_loss2 = test_loss / len(test_loader)
test_accuracy2 = 100.0 * correct2 / total2

print(f"Test Loss: {avg_test_loss2:.4f}")
print(f"Test Accuracy: {test_accuracy2:.2f}%")


In [ ]:
epochs2 = range(1, len(train_losses2) + 1)

plt.figure(figsize=(8, 6))

plt.plot(epochs2, train_losses2, label="Training Loss", marker="o")
plt.plot(epochs2, val_losses2, label="Validation Loss", marker="o")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
MODEL_PATH = "fruits_classifier_global_pool_regularize.pth"

torch.save({
    "model_state_dict": model2.state_dict(),
    "class_to_idx": train_dataset.class_to_idx
}, MODEL_PATH)

print("Model saved successfully!")

#### Optuna - Hyperparameter Tuning

In [ ]:
import optuna

def objective(trial):
    # ---- hyperparameter to tune (ONLY ONE) ----
    weight_decay = trial.suggest_float(
        "weight_decay", 1e-5, 1e-3, log=True
    )

    # ---- model ----
    model3 = NextCNN(num_classes=2).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer3 = optim.AdamW(
        model3.parameters(),
        lr=0.001,              # fixed
        weight_decay=weight_decay
    )

    epochs = 5  

    # ---- training + validation ----
    for epoch in range(epochs):
        model3.train()
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer3.zero_grad()
            outputs3 = model3(images)
            loss = criterion(outputs3, labels)
            loss.backward()
            optimizer3.step()

    # ---- validation accuracy (objective metric) ----
    model3.eval()
    correct3 = 0
    total3 = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs3 = model3(images)
            _, preds = torch.max(outputs3, dim=1)

            total3 += labels.size(0)
            correct3 += (preds == labels).sum().item()

    val_accuracy = correct3 / total3

    return val_accuracy


In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=3)

In [ ]:
print("Best trial:")
print(f"  Validation Accuracy: {study.best_value:.4f}")
print("  Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

In [ ]:
best_weight_decay = study.best_params["weight_decay"]
print("Best weight_decay:", best_weight_decay)

#### Transfer Learning approach

In [16]:
from torchvision import models
import torch.nn as nn

# Load pretrained ResNet-50
model_tl = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze all convolutional layers
for param in model_tl.parameters():
    param.requires_grad = False

# Replace final fully connected layer
num_ftrs = model_tl.fc.in_features
model_tl.fc = nn.Linear(num_ftrs, 2)  # fresh vs spoiled

model_tl = model_tl.to(device)

In [17]:
criterion_tl = nn.CrossEntropyLoss()
optimizer_tl = torch.optim.Adam(
    model_tl.fc.parameters()  # ONLY final layer. Use default learning rate
)


In [18]:
train_losses_tl, val_losses_tl, true_labels_tl, pred_labels_tl = train_and_validate(
    model=model_tl,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion_tl,
    optimizer=optimizer_tl,
    device=device,
    epochs=5
)


Epoch [1/5] | Train Loss: 0.2982 | Val Loss: 0.1701 | Val Acc: 94.58%
Epoch [2/5] | Train Loss: 0.1529 | Val Loss: 0.1168 | Val Acc: 96.29%
Epoch [3/5] | Train Loss: 0.1140 | Val Loss: 0.0954 | Val Acc: 96.96%
Epoch [4/5] | Train Loss: 0.1021 | Val Loss: 0.0834 | Val Acc: 97.29%
Epoch [5/5] | Train Loss: 0.0917 | Val Loss: 0.0735 | Val Acc: 97.46%


In [19]:
MODEL_PATH = "fruits_classifier_resent50_tl.pth"

torch.save({
    "model_state_dict": model_tl.state_dict(),
    "class_to_idx": train_dataset.class_to_idx
}, MODEL_PATH)

print("Model saved successfully!")

Model saved successfully!
